In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

# --- CONFIGURATION: THE JUMIA MAP ---
# We define categories and their base URLs for page 1
JUMIA_CATEGORIES = {
    "Appliances": {
        "Microwaves": "https://www.jumia.ug/small-appliances-microwave/",
        "Refrigerators": "https://www.jumia.ug/freezers/",
        "Blenders": "https://www.jumia.ug/blenders/"
        # Add others here...
    },
    "Computing": {
        "Laptops": "https://www.jumia.ug/laptops/",
        "Printers": "https://www.jumia.ug/printers-scanners/",
        "Phones": "https://www.jumia.ug/phones-tablets/",
        "Electronics":"https://www.jumia.ug/catalog/?q=electronics",
        # Add others here...
    },
    "Home & Office": {
        "Furniture": "https://www.jumia.ug/furniture/",
        "Home Decor": "https://www.jumia.ug/home-decor/",
        "Kitchen & Dining": "https://www.jumia.ug/kitchen-dining/"
        # Add others here...
    },
    
    # Add the rest of your categories map here...
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",    
    "Accept-Language": "en-US,en;q=0.9",
}

# To store all scraped data
all_products_data = []

In [2]:
def scrape_category_page(url, main_cat, sub_cat):
    """Scrapes one page of listings and attempts to find the next page."""
    print(f"--- Scraping: {sub_cat} in {main_cat} ---")
    
    try:
        # Be polite
        time.sleep(random.uniform(2, 4))
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        # Jumia product listings are usually inside articles with this class
        products = soup.find_all("article", class_="prd")
        
        print(f"Found {len(products)} products on this page.")

        for card in products:
            # 1. Name
            # Often found in an h3 with class 'name'
            name_element = card.find("h3", class_="name")
            name = name_element.get_text().strip() if name_element else "Unknown"

            # 2. Price
            # Often found in div with class 'prc'
            price_element = card.find("div", class_="prc")
            if price_element:
                price_str = price_element.get_text().strip()
                # Clean: remove "USh", commas
                price = int(price_str.replace("USh", "").replace(",", "").strip())
            else:
                price = 0

            # 3. Rating (Stars)
            # Often found in div class 'rev' (stars out of 5)
            rating_element = card.find("div", class_="rev")
            if rating_element:
                # Jumia ratings are sometimes text like "4.1 out of 5"
                rating_text = rating_element.get_text().split(" ")[0]
                try:
                    rating = float(rating_text)
                except ValueError:
                    rating = 0.0
            else:
                rating = 0.0

            all_products_data.append({
                "Main_Category": main_cat,
                "Sub_Category": sub_cat,
                "Product_Name": name,
                "Price_UGX": price,
                "Rating": rating
            })

        # --- PAGINATION LOGIC ---
        # Look for the 'Next' link at the bottom of the page
        # Usually an 'a' tag with aria-label="Next Page"
        next_button_container = soup.find("a", {"aria-label": "Next Page"})
        
        if next_button_container:
            # Get the relative href and make it absolute
            relative_url = next_button_container.get('href')
            next_url = "https://www.jumia.ug" + relative_url
            return next_url # Return the URL for the next page to scrape
        else:
            return None # No more pages

    except Exception as e:
        print(f"Error scraping page {url}: {e}")
        return None # Stop on error
    
# --- MAIN EXECUTION ---
if __name__ == "__main__":
    
    # Loop through categories map
    for main_category, sub_categories in JUMIA_CATEGORIES.items():
        for sub_category, base_url in sub_categories.items():
            
            # Start with page 1
            current_url = base_url
            
            # Keep scraping until scrape_category_page returns None
            while current_url:
                next_page_url = scrape_category_page(current_url, main_category, sub_category)
                current_url = next_page_url # Update URL to the next page
                
                # Safety break: only scrape first 2 pages per sub-category for testing
                # Remove this 'break' later when you want to scrape everything
                # break 
                
                # Optional: Add a longer sleep between pages to avoid getting blocked
                if current_url:
                    time.sleep(random.uniform(5, 10))

    # --- SAVE DATA ---
    print("Scraping complete. Saving data...")
    if all_products_data:
        df = pd.DataFrame(all_products_data)
        
        # Remove products with 0 price (usually errors)
        df = df[df['Price_UGX'] > 0]
        
        df.to_csv("jumia_market_data.csv", index=False)
        print(f"Successfully saved {len(df)} products to jumia_market_data.csv")
    else:
        print("No data was scraped.")

--- Scraping: Microwaves in Appliances ---
Error scraping page https://www.jumia.ug/small-appliances-microwave/: 403 Client Error: Forbidden for url: https://www.jumia.ug/small-appliances-microwave/
--- Scraping: Refrigerators in Appliances ---
Error scraping page https://www.jumia.ug/freezers/: 403 Client Error: Forbidden for url: https://www.jumia.ug/freezers/
--- Scraping: Blenders in Appliances ---
Error scraping page https://www.jumia.ug/blenders/: 403 Client Error: Forbidden for url: https://www.jumia.ug/blenders/
--- Scraping: Laptops in Computing ---
Error scraping page https://www.jumia.ug/laptops/: 403 Client Error: Forbidden for url: https://www.jumia.ug/laptops/
--- Scraping: Printers in Computing ---
Error scraping page https://www.jumia.ug/printers-scanners/: 403 Client Error: Forbidden for url: https://www.jumia.ug/printers-scanners/
--- Scraping: Phones in Computing ---
Error scraping page https://www.jumia.ug/phones-tablets/: 403 Client Error: Forbidden for url: https: